# 04a — Métriques de régression (protocole §1, §2, §3)

Applique au XGBoost de la Phase 3 les sections 1 à 3 de [`docs/protocole_evaluation.md`](../docs/protocole_evaluation.md) : 5 baselines, 10 métriques de régression, erreur par tranche de RUL, sur train, validation (par pli) et test officiel.

**Règles de ce notebook** : le test officiel n'est utilisé que pour le tableau final, jamais pour choisir quoi que ce soit. Les plis sont calculés une fois et réutilisés partout.

## 1. Reconstruire la référence de la Phase 3

Avant de comparer quoi que ce soit, on reconstruit exactement le pipeline de `03_modelisation.ipynb` (mêmes 46 features, mêmes 5 plis) et on vérifie qu'on retrouve son RMSE de validation (16.86). Si ce chiffre ne colle pas, toute la suite est invalide.

In [ ]:
import sys
sys.path.insert(0, "..")  # pour pouvoir importer src/ depuis notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor

from src.preprocessing import (
    load_fd001, get_sensor_columns, find_constant_sensors,
    normalize_minmax, lisser_capteurs, calculer_rul, plafonner_rul, DATA_DIR,
)

FENETRE = 5        # taille de la fenetre glissante (Phase 1)
PLAFOND_RUL = 125  # plafond de la RUL (Phase 2)
# reglages XGBoost de la Phase 3, repris tels quels
PARAMS_XGB = dict(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)

# --- Chargement et preparation du train (memes etapes que la Phase 3) ---
train = load_fd001("train")  # fichier d'entrainement FD001
sensor_cols = get_sensor_columns(train.columns)  # les 21 capteurs bruts
constant_sensors = find_constant_sensors(train, sensor_cols)  # capteurs constants (6)
train = train.drop(columns=constant_sensors)  # on les retire
sensor_cols_restants = get_sensor_columns(train.columns)  # les 15 capteurs utiles

train, scaler = normalize_minmax(train, sensor_cols_restants)  # min-max 0 a 1 (scaler garde pour le test)
train = lisser_capteurs(train, sensor_cols_restants, fenetre=FENETRE)  # moyenne mobile causale par moteur
train = calculer_rul(train)  # RUL brute
train = plafonner_rul(train, plafond=PLAFOND_RUL)  # RUL plafonnee a 125


def ajouter_features_glissantes(df, sensor_cols, fenetre):
    """
    Ajoute, pour chaque capteur, la moyenne et l'ecart-type des `fenetre` derniers
    cycles, moteur par moteur (causal). Copie de la fonction du notebook 03.

    Parameters
    ----------
    df : pandas.DataFrame
        Tableau source (une copie est renvoyee).
    sensor_cols : list de str
        Capteurs pour lesquels calculer les features.
    fenetre : int
        Nombre de cycles de la fenetre glissante.

    Returns
    -------
    pandas.DataFrame
        Tableau avec "<capteur>_glissante_moy" et "<capteur>_glissante_std" en plus.
    """
    df = df.copy()  # copie independante

    for sensor_name in sensor_cols:  # un capteur a la fois
        colonne_moyenne = sensor_name + "_glissante_moy"
        colonne_ecart_type = sensor_name + "_glissante_std"
        df[colonne_moyenne] = 0.0  # valeurs provisoires
        df[colonne_ecart_type] = 0.0

        for moteur_id in df["unit_number"].unique():  # un moteur a la fois
            masque = df["unit_number"] == moteur_id  # lignes de ce moteur
            df.loc[masque, colonne_moyenne] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).mean()
            df.loc[masque, colonne_ecart_type] = df.loc[masque, sensor_name].rolling(fenetre, min_periods=1).std()

    return df


train = ajouter_features_glissantes(train, sensor_cols_restants, FENETRE)

# noms des colonnes moyenne / ecart-type glissants, trouves par boucle explicite
colonnes_moyenne = []
colonnes_ecart_type = []
for colonne in train.columns:
    if colonne.endswith("_glissante_moy"):
        colonnes_moyenne.append(colonne)
    if colonne.endswith("_glissante_std"):
        colonnes_ecart_type.append(colonne)

# au 1er cycle de chaque moteur l'ecart-type n'existe pas (1 seule valeur) : NaN remplaces par 0
train[colonnes_ecart_type] = train[colonnes_ecart_type].fillna(0.0)

# 15 capteurs + 15 moyennes + 15 ecarts-types + time_cycles = 46 features
feature_cols = sensor_cols_restants + colonnes_moyenne + colonnes_ecart_type + ["time_cycles"]
assert len(feature_cols) == 46

X = train[feature_cols]        # entrees du modele
y = train["RUL_cappee"]        # cible plafonnee
groups = train["unit_number"]  # quel moteur : sert a GroupKFold seulement

# --- Plis : calcules UNE fois, reutilises par tous les modeles (protocole §1.1 et §1.2) ---
plis = list(GroupKFold(n_splits=5, shuffle=True, random_state=42).split(X, y, groups=groups))
for numero, (idx_train, idx_val) in enumerate(plis):
    moteurs_train = set(groups.iloc[idx_train])
    moteurs_val = set(groups.iloc[idx_val])
    assert len(moteurs_train & moteurs_val) == 0, f"pli {numero + 1} : un moteur est des deux cotes"


def score_nasa(y_true, y_pred):
    """
    Score NASA (Saxena et al., 2008) : somme des penalites, asymetrique
    (surestimer la RUL coute plus cher que la sous-estimer).

    Parameters
    ----------
    y_true : array-like
        RUL vraies.
    y_pred : array-like
        RUL predites.

    Returns
    -------
    float
        Somme des penalites sur tous les exemples.
    """
    total = 0.0  # on cumule la penalite de chaque exemple
    for vrai, predit in zip(y_true, y_pred):
        d = predit - vrai  # positif = surestimation
        if d < 0:
            total += np.exp(-d / 13) - 1  # sous-estimation : penalite douce
        else:
            total += np.exp(d / 10) - 1   # surestimation : penalite severe
    return total


# --- Verification : on doit retrouver le RMSE de validation de la Phase 3 (16.86) ---
rmse_val_plis = []
for idx_train, idx_val in plis:
    modele = XGBRegressor(**PARAMS_XGB)  # modele neuf a chaque pli
    modele.fit(X.iloc[idx_train], y.iloc[idx_train])
    pred_val = modele.predict(X.iloc[idx_val])
    rmse_val_plis.append(np.sqrt(np.mean((y.iloc[idx_val].values - pred_val) ** 2)))

print("RMSE validation par pli :", [round(r, 2) for r in rmse_val_plis])
print(f"Moyenne : {np.mean(rmse_val_plis):.2f} (reference Phase 3 : 16.86)")
assert abs(np.mean(rmse_val_plis) - 16.86) < 0.1, "on ne retrouve pas la reference de la Phase 3"

## 2. Les 5 baselines (protocole §2) et les prédictions pli par pli

Chaque référence est entraînée sur les moteurs d'entraînement de chaque pli et évaluée sur les moteurs de validation du même pli (mêmes `plis` que XGBoost). On garde les prédictions train **et** validation de chaque pli : toutes les métriques des sections suivantes sont calculées à partir de ces mêmes valeurs.

| Référence | Implémentation | Choix documenté |
|---|---|---|
| Moyenne | `DummyRegressor(strategy="mean")` | moyenne de la RUL plafonnée du train du pli |
| Médiane | `DummyRegressor(strategy="median")` | médiane du train du pli |
| Constante 125 | `DummyRegressor(strategy="constant", constant=125)` | « le moteur est toujours en pleine santé » |
| Aléatoire | classe `AleatoireUniforme` ci-dessous | tirage uniforme [0, 125], graine `42 + numéro du pli` (reproductible, et deux plis ne reçoivent pas exactement le même bruit) |
| Ridge | `Pipeline(StandardScaler, Ridge(alpha=1.0))` | `StandardScaler` ajusté **dans chaque pli** (`time_cycles` va jusqu'à plus de 300, les capteurs sont entre 0 et 1 : sans standardisation la pénalité de Ridge ne s'applique pas équitablement). `alpha=1.0` = valeur par défaut, volontairement non réglé : c'est une baseline, pas un modèle optimisé |
| XGBoost | `XGBRegressor(**PARAMS_XGB)` | le modèle de la Phase 3, réglages inchangés |

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import r2_score


class AleatoireUniforme(BaseEstimator, RegressorMixin):
    """
    Modele "completement rate" : predit un nombre tire au hasard, uniformement
    entre borne_basse et borne_haute, sans regarder les capteurs.

    Parameters
    ----------
    borne_basse : float
        Plus petite valeur possible de la prediction.
    borne_haute : float
        Plus grande valeur possible de la prediction.
    graine : int
        Graine du generateur aleatoire (resultats reproductibles).
    """

    def __init__(self, borne_basse=0.0, borne_haute=125.0, graine=42):
        self.borne_basse = borne_basse  # borne inferieure du tirage
        self.borne_haute = borne_haute  # borne superieure du tirage
        self.graine = graine            # graine fixe

    def fit(self, X, y):
        """N'apprend rien (le modele ignore les donnees) ; renvoie self par convention scikit-learn."""
        return self

    def predict(self, X):
        """Renvoie un tirage uniforme par ligne de X."""
        generateur = np.random.default_rng(self.graine)  # generateur initialise avec la graine
        return generateur.uniform(self.borne_basse, self.borne_haute, size=len(X))


def fabriquer_modele(nom, numero_pli):
    """
    Cree un modele NEUF pour la reference `nom` (aucune memoire d'un pli a l'autre).

    Parameters
    ----------
    nom : str
        Nom de la reference (element de NOMS_MODELES).
    numero_pli : int
        Numero du pli (0 a 4), utilise seulement pour la graine du modele aleatoire.

    Returns
    -------
    objet scikit-learn / XGBoost avec fit() et predict().
    """
    if nom == "Moyenne":
        return DummyRegressor(strategy="mean")
    if nom == "Mediane":
        return DummyRegressor(strategy="median")
    if nom == "Constante 125":
        return DummyRegressor(strategy="constant", constant=PLAFOND_RUL)
    if nom == "Aleatoire U(0,125)":
        return AleatoireUniforme(0.0, float(PLAFOND_RUL), graine=42 + numero_pli)
    if nom == "Ridge":
        return Pipeline([("standardisation", StandardScaler()), ("ridge", Ridge(alpha=1.0))])
    if nom == "XGBoost (46 features)":
        return XGBRegressor(**PARAMS_XGB)
    raise ValueError(f"modele inconnu : {nom}")


NOMS_MODELES = ["Moyenne", "Mediane", "Constante 125", "Aleatoire U(0,125)", "Ridge", "XGBoost (46 features)"]

# predictions[nom] = liste de 5 dicts (un par pli) avec y_tr, pred_tr, y_val, pred_val
predictions = {}
for nom in NOMS_MODELES:  # une reference a la fois
    predictions[nom] = []
    for numero_pli, (idx_train, idx_val) in enumerate(plis):  # les 5 memes plis
        modele = fabriquer_modele(nom, numero_pli)  # modele neuf
        modele.fit(X.iloc[idx_train], y.iloc[idx_train])  # apprend sur les moteurs d'entrainement
        predictions[nom].append({
            "y_tr": y.iloc[idx_train].values,
            "pred_tr": modele.predict(X.iloc[idx_train]),   # moteurs appris
            "y_val": y.iloc[idx_val].values,
            "pred_val": modele.predict(X.iloc[idx_val]),    # moteurs jamais vus
        })
    print(f"{nom:24s} : 5 plis entraines")

# sanity check : la baseline "moyenne" doit avoir un R2 de validation proche de 0 (par construction)
r2_moyenne = []
for pli_pred in predictions["Moyenne"]:
    r2_moyenne.append(r2_score(pli_pred["y_val"], pli_pred["pred_val"]))
print(f"R2 validation moyen de la baseline Moyenne : {np.mean(r2_moyenne):.3f} (attendu ~ 0)")
assert abs(np.mean(r2_moyenne)) < 0.05

## 3. Les 10 métriques de régression (protocole §3)

Une seule fonction, `calculer_metriques`, calcule toutes les métriques à partir de `(y_true, y_pred)`. Elle est utilisée à l'identique pour train, validation et test.

Conventions : biais = `mean(prédit − vrai)` (positif = surestimation, le sens dangereux) ; score NASA en **somme** et en **moyenne par prédiction** ; « surestimations > 20 » = nombre (et %) de lignes avec `prédit − vrai > 20` ; MAPE calculée telle quelle par `sklearn` (qui remplace un dénominateur nul par un epsilon minuscule, d'où des valeurs gigantesques quand la RUL vraie vaut 0). L'écart-type entre plis est `np.std` (ddof=0), comme dans la Phase 3.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, median_absolute_error,
    max_error, mean_absolute_percentage_error,
)

NOMS_METRIQUES = ["MAE", "MSE", "RMSE", "R2", "MedAE", "MaxErr", "Biais",
                  "NASA_somme", "NASA_moy", "Sur20_n", "Sur20_pct", "MAPE"]


def calculer_metriques(y_true, y_pred):
    """
    Calcule toutes les metriques de regression du protocole §3.

    Parameters
    ----------
    y_true : array-like
        RUL vraies.
    y_pred : array-like
        RUL predites.

    Returns
    -------
    dict
        Une valeur par nom de NOMS_METRIQUES.
    """
    y_true = np.asarray(y_true, dtype=float)  # tableaux numpy de flottants
    y_pred = np.asarray(y_pred, dtype=float)
    ecarts = y_pred - y_true  # positif = surestimation (dangereux)
    nombre = len(y_true)
    mse = mean_squared_error(y_true, y_pred)
    nasa_somme = score_nasa(y_true, y_pred)
    nombre_sur20 = int(np.sum(ecarts > 20))  # surestimations de plus de 20 cycles

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred),
        "MaxErr": max_error(y_true, y_pred),
        "Biais": np.mean(ecarts),
        "NASA_somme": nasa_somme,
        "NASA_moy": nasa_somme / nombre,
        "Sur20_n": nombre_sur20,
        "Sur20_pct": 100.0 * nombre_sur20 / nombre,
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }


# sanity check sur un cas calculable a la main : vrai = [10, 20], predit = [15, 15]
verif = calculer_metriques([10, 20], [15, 15])
assert verif["MAE"] == 5.0 and verif["MSE"] == 25.0 and verif["RMSE"] == 5.0
assert verif["Biais"] == 0.0 and verif["MaxErr"] == 5.0 and verif["Sur20_n"] == 0
assert abs(verif["NASA_somme"] - ((np.exp(0.5) - 1) + (np.exp(5 / 13) - 1))) < 1e-9
# et on retrouve le RMSE de XGBoost de la section 1
verif_xgb = []
for pli_pred in predictions["XGBoost (46 features)"]:
    verif_xgb.append(calculer_metriques(pli_pred["y_val"], pli_pred["pred_val"])["RMSE"])
assert abs(np.mean(verif_xgb) - np.mean(rmse_val_plis)) < 0.05

# metriques de chaque modele, de chaque pli, sur train et sur validation
metriques_plis = {}  # metriques_plis[nom][jeu] = liste de 5 dicts (un par pli)
for nom in NOMS_MODELES:
    metriques_plis[nom] = {"train": [], "validation": []}
    for pli_pred in predictions[nom]:
        metriques_plis[nom]["train"].append(calculer_metriques(pli_pred["y_tr"], pli_pred["pred_tr"]))
        metriques_plis[nom]["validation"].append(calculer_metriques(pli_pred["y_val"], pli_pred["pred_val"]))
print("Metriques calculees pour", len(NOMS_MODELES), "modeles x 2 jeux x 5 plis.")

## 4. Tableaux train et validation (moyenne ± écart-type entre plis)

Validation : valeur par pli, puis moyenne ± écart-type. Train : moyenne des plis. Un écart entre deux modèles plus petit que l'écart-type entre plis n'est pas une différence établie (protocole §8 ; le test statistique lui-même est hors périmètre de ce notebook).

In [ ]:
def formater(valeur, nom_metrique):
    """Met en forme un nombre selon la metrique (grands nombres sans decimale, MAPE en notation scientifique)."""
    if nom_metrique == "MAPE":
        return f"{valeur:.3g}"
    if nom_metrique in ("NASA_somme", "MSE"):
        return f"{valeur:,.0f}".replace(",", " ")
    return f"{valeur:.2f}"


def tableau_synthese(jeu, avec_ecart_type):
    """
    Construit un DataFrame (modeles en lignes, metriques en colonnes) avec la moyenne entre plis,
    suivie de ' ± ecart-type' si avec_ecart_type est vrai.
    """
    lignes = {}
    for nom in NOMS_MODELES:
        ligne = {}
        for metrique in NOMS_METRIQUES:
            valeurs_plis = []
            for metriques_un_pli in metriques_plis[nom][jeu]:
                valeurs_plis.append(metriques_un_pli[metrique])
            texte = formater(np.mean(valeurs_plis), metrique)
            if avec_ecart_type:
                texte = texte + " ± " + formater(np.std(valeurs_plis), metrique)
            ligne[metrique] = texte
        lignes[nom] = ligne
    return pd.DataFrame(lignes).T


print("=== VALIDATION : moyenne ± ecart-type entre les 5 plis ===")
tableau_validation = tableau_synthese("validation", True)
print(tableau_validation.to_string())
print("\n=== TRAIN : moyenne des 5 plis ===")
tableau_train = tableau_synthese("train", False)
print(tableau_train.to_string())

# detail par pli de la validation pour Ridge et XGBoost
print("\n=== VALIDATION par pli ===")
for nom in ["Ridge", "XGBoost (46 features)"]:
    for numero_pli in range(5):
        m = metriques_plis[nom]["validation"][numero_pli]
        print(f"{nom:24s} pli {numero_pli + 1} : RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R2={m['R2']:.3f}  NASA_moy={m['NASA_moy']:.1f}")

## 5. Erreur par tranche de RUL (protocole §3)

MAE, RMSE et biais séparément pour RUL vraie ≤ 30, 31-60 et 61-125. En validation, on regroupe les prédictions des 5 plis (chaque ligne du train est prédite exactement une fois, par un modèle qui ne l'a pas vue) : les tranches sont ainsi assez remplies. Le RMSE global mélange le début de vie (RUL plafonnée, tâche facile) et la fin de vie, seule zone où une décision de maintenance se prend.

In [ ]:
TRANCHES = [("<= 30", 0, 30), ("31-60", 31, 60), ("61-125", 61, 125)]  # (nom, borne basse, borne haute incluses)


def metriques_par_tranche(y_true, y_pred, tranches):
    """
    MAE, RMSE, biais et effectif pour chaque tranche de RUL VRAIE.

    Parameters
    ----------
    y_true, y_pred : array-like
        RUL vraies et predites.
    tranches : liste de (nom, borne_basse, borne_haute)
        Bornes incluses. Les valeurs de y_true hors de toutes les tranches sont ignorees.

    Returns
    -------
    dict : nom de tranche -> dict(N, MAE, RMSE, Biais)
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    resultat = {}
    for nom_tranche, bas, haut in tranches:
        masque = (y_true >= bas) & (y_true <= haut)  # lignes dont la RUL vraie est dans la tranche
        if masque.sum() == 0:
            continue  # tranche vide : rien a calculer
        ecarts = y_pred[masque] - y_true[masque]
        resultat[nom_tranche] = {
            "N": int(masque.sum()),
            "MAE": np.mean(np.abs(ecarts)),
            "RMSE": np.sqrt(np.mean(ecarts ** 2)),
            "Biais": np.mean(ecarts),
        }
    return resultat


def afficher_tranches(titre, y_true_par_modele, y_pred_par_modele, tranches):
    """Affiche un tableau (modele x tranche) de N, MAE, RMSE, biais."""
    print(titre)
    print(f"{'modele':24s} {'tranche':8s} {'N':>6s} {'MAE':>7s} {'RMSE':>7s} {'biais':>7s}")
    for nom in NOMS_MODELES:
        par_tranche = metriques_par_tranche(y_true_par_modele[nom], y_pred_par_modele[nom], tranches)
        for nom_tranche, valeurs in par_tranche.items():
            print(f"{nom:24s} {nom_tranche:8s} {valeurs['N']:6d} {valeurs['MAE']:7.2f} {valeurs['RMSE']:7.2f} {valeurs['Biais']:+7.2f}")


# predictions de validation regroupees (5 plis mis bout a bout), pour chaque modele
y_val_regroupe = {}
pred_val_regroupe = {}
for nom in NOMS_MODELES:
    morceaux_vrai = []
    morceaux_pred = []
    for pli_pred in predictions[nom]:
        morceaux_vrai.append(pli_pred["y_val"])
        morceaux_pred.append(pli_pred["pred_val"])
    y_val_regroupe[nom] = np.concatenate(morceaux_vrai)
    pred_val_regroupe[nom] = np.concatenate(morceaux_pred)

# controle : les 5 plis couvrent bien les 20 631 lignes exactement une fois
assert len(y_val_regroupe["XGBoost (46 features)"]) == len(train)

afficher_tranches("=== VALIDATION (5 plis regroupes) ===", y_val_regroupe, pred_val_regroupe, TRANCHES)

# diagnostic MAPE : par tranche, pour XGBoost, pour voir d'ou vient l'explosion
print("\nMAPE de XGBoost en validation, par tranche de RUL vraie :")
y_v = y_val_regroupe["XGBoost (46 features)"]
p_v = pred_val_regroupe["XGBoost (46 features)"]
for nom_tranche, bas, haut in [("= 0", 0, 0), ("1-10", 1, 10), ("11-30", 11, 30), ("31-60", 31, 60), ("61-125", 61, 125)]:
    masque = (y_v >= bas) & (y_v <= haut)
    print(f"  RUL vraie {nom_tranche:7s} (N={masque.sum():5d}) : MAPE = {mean_absolute_percentage_error(y_v[masque], p_v[masque]):.3g}")

## 6. Test officiel (utilisé une seule fois, pour le tableau final)

Rien n'est choisi ici : les modèles et leurs réglages sont ceux des sections précédentes. Chaque référence est réentraînée sur **les 100 moteurs du train** puis évaluée sur le **dernier cycle de chacun des 100 moteurs du test** (`test_FD001.txt`), comparé à `RUL_FD001.txt` (RUL vraie **non plafonnée**). Même prétraitement que la Phase 3 : le `scaler` du train est appliqué en `transform` seul. Comme en Phase 3, les prédictions sont ramenées à 0 si négatives (une RUL négative n'a pas de sens physique) ; ça concerne tous les modèles, y compris les baselines et Ridge.

Rappel (protocole §1.5) : validation = toutes les lignes des moteurs mis de côté, cible plafonnée ; test = 1 prédiction par moteur, vérité non plafonnée. Les sommes de score NASA ne sont pas comparables entre les deux ; on compare les **moyennes par prédiction**. La tranche « 126+ » n'existe qu'au test (RUL vraie non plafonnée) et est affichée à part.

In [ ]:
test = load_fd001("test")  # fichier de test FD001
test = test.drop(columns=constant_sensors)  # memes capteurs retires que sur le train
test[sensor_cols_restants] = scaler.transform(test[sensor_cols_restants])  # transform seul : pas de fuite
test = lisser_capteurs(test, sensor_cols_restants, fenetre=FENETRE)
test = ajouter_features_glissantes(test, sensor_cols_restants, FENETRE)
test[colonnes_ecart_type] = test[colonnes_ecart_type].fillna(0.0)

# dernier cycle observe de chaque moteur test
indices_derniers_cycles = test.groupby("unit_number")["time_cycles"].idxmax()
dernier_cycle_par_moteur = test.loc[indices_derniers_cycles].sort_values("unit_number").reset_index(drop=True)
rul_test_vraie = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL_vraie"])
assert dernier_cycle_par_moteur.shape[0] == 100 and rul_test_vraie.shape[0] == 100

X_test = dernier_cycle_par_moteur[feature_cols]
y_test_vraie = rul_test_vraie["RUL_vraie"].values.astype(float)

# chaque reference est reentrainee sur tout le train (100 moteurs), puis predit le test
y_test_par_modele = {}
pred_test_par_modele = {}
metriques_test = {}
for nom in NOMS_MODELES:
    modele_final_ref = fabriquer_modele(nom, 0)  # graine du modele aleatoire : 42 (pli 0)
    modele_final_ref.fit(X, y)
    prediction = np.clip(modele_final_ref.predict(X_test), 0, None)  # RUL negative impossible
    y_test_par_modele[nom] = y_test_vraie
    pred_test_par_modele[nom] = prediction
    metriques_test[nom] = calculer_metriques(y_test_vraie, prediction)

# controle : on doit retrouver les chiffres publies de la Phase 3 pour XGBoost
m_xgb = metriques_test["XGBoost (46 features)"]
print(f"XGBoost test : RMSE={m_xgb['RMSE']:.2f} (Phase 3 : 19.70), score NASA={m_xgb['NASA_somme']:.1f} (852.9), surestimes>20={m_xgb['Sur20_n']}/100 (9)")
assert abs(m_xgb["RMSE"] - 19.70) < 0.1 and abs(m_xgb["NASA_somme"] - 852.9) < 5

# tableau test : une ligne par modele
lignes_test = {}
for nom in NOMS_MODELES:
    ligne = {}
    for metrique in NOMS_METRIQUES:
        ligne[metrique] = formater(metriques_test[nom][metrique], metrique)
    lignes_test[nom] = ligne
print("\n=== TEST OFFICIEL (100 moteurs, 1 prediction par moteur) ===")
print(pd.DataFrame(lignes_test).T.to_string())

TRANCHES_TEST = TRANCHES + [("126+", 126, 10000)]  # la RUL vraie du test n'est pas plafonnee
effectifs = {}
for nom_tranche, bas, haut in TRANCHES_TEST:
    effectifs[nom_tranche] = int(np.sum((y_test_vraie >= bas) & (y_test_vraie <= haut)))
print("\nRepartition de la RUL vraie du test :", effectifs)
afficher_tranches("\n=== TEST (tranches) ===", y_test_par_modele, pred_test_par_modele, TRANCHES_TEST)

## 7. Figures de régression (protocole §9)

RUL prédite vs vraie (validation et test, avec la diagonale idéale) et résidus (prédit − vrai) selon la RUL vraie. Enregistrées dans `docs/figures/` avec le préfixe `04a_`. Validation : prédictions des 5 plis regroupées, ~20 600 points, d'où la transparence ; ligne rouge = moyenne des résidus par paquet de 10 cycles.

In [ ]:
NOM_XGB = "XGBoost (46 features)"


def moyenne_par_paquet(x, valeurs, largeur=10):
    """Moyenne de `valeurs` par paquet de `largeur` cycles de x (pour tracer la tendance des residus)."""
    centres = []
    moyennes = []
    for debut in range(0, int(np.max(x)) + 1, largeur):  # 0-9, 10-19, ...
        masque = (x >= debut) & (x < debut + largeur)
        if masque.sum() >= 5:  # on ignore les paquets presque vides
            centres.append(debut + largeur / 2)
            moyennes.append(np.mean(valeurs[masque]))
    return centres, moyennes


# --- Figure 1 : predit vs vrai (validation | test) ---
rmse_val_xgb = np.mean(rmse_val_plis)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].scatter(y_val_regroupe[NOM_XGB], pred_val_regroupe[NOM_XGB], s=4, alpha=0.15)
axes[0].plot([0, 130], [0, 130], "--", color="black", label="prédiction = vérité")
axes[0].set_title(f"Validation croisée (5 plis regroupés) — RMSE moyen {rmse_val_xgb:.2f}")
axes[0].set_xlabel("RUL vraie (plafonnée à 125, cycles)")
axes[0].set_ylabel("RUL prédite (cycles)")
axes[0].legend()

axes[1].scatter(y_test_vraie, pred_test_par_modele[NOM_XGB], alpha=0.7)
limite_test = max(y_test_vraie.max(), pred_test_par_modele[NOM_XGB].max()) + 5
axes[1].plot([0, limite_test], [0, limite_test], "--", color="black", label="prédiction = vérité")
axes[1].set_title(f"Test officiel (100 moteurs) — RMSE {m_xgb['RMSE']:.2f}")
axes[1].set_xlabel("RUL vraie (non plafonnée, cycles)")
axes[1].set_ylabel("RUL prédite (cycles)")
axes[1].legend()
plt.suptitle("XGBoost : RUL prédite vs RUL vraie")
plt.tight_layout()
plt.savefig("../docs/figures/04a_predit_vs_vrai_validation_test.png", dpi=120)
plt.show()

# --- Figure 2 : residus selon la RUL vraie (validation | test) ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
residus_val = pred_val_regroupe[NOM_XGB] - y_val_regroupe[NOM_XGB]
axes[0].scatter(y_val_regroupe[NOM_XGB], residus_val, s=4, alpha=0.15)
centres, moyennes = moyenne_par_paquet(y_val_regroupe[NOM_XGB], residus_val)
axes[0].plot(centres, moyennes, color="red", marker="o", label="moyenne par paquet de 10 cycles")
axes[0].axhline(0, color="black", linestyle="--")
axes[0].axhline(20, color="grey", linestyle=":", label="surestimation de 20 cycles")
axes[0].set_title("Validation croisée")
axes[0].set_xlabel("RUL vraie (cycles)")
axes[0].set_ylabel("Résidu = prédit − vrai (cycles) ; > 0 : surestimation")
axes[0].legend()

residus_test = pred_test_par_modele[NOM_XGB] - y_test_vraie
axes[1].scatter(y_test_vraie, residus_test, alpha=0.7)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].axhline(20, color="grey", linestyle=":", label="surestimation de 20 cycles")
axes[1].set_title("Test officiel (100 moteurs)")
axes[1].set_xlabel("RUL vraie (non plafonnée, cycles)")
axes[1].set_ylabel("Résidu = prédit − vrai (cycles)")
axes[1].legend()
plt.suptitle("XGBoost : résidus selon la RUL vraie")
plt.tight_layout()
plt.savefig("../docs/figures/04a_residus_selon_rul_vraie.png", dpi=120)
plt.show()

## 8. XGBoost contre Ridge, pli par pli (description, pas encore un test de significativité)

Le protocole (§2) demande si XGBoost bat clairement `Ridge`. On regarde la différence **pli par pli** (mêmes moteurs de validation des deux côtés) : si le signe est le même sur les 5 plis, ce n'est pas un effet du découpage. Le test statistique formel (t apparié / Wilcoxon, §8) n'est pas fait ici : avec 5 plis il a très peu de puissance et il relève de la comparaison entre modèles.

In [ ]:
print("Difference (Ridge - XGBoost) par pli ; positif = XGBoost meilleur (erreur plus faible)")
print(f"{'metrique':10s} " + " ".join([f"pli {i + 1:d}" .rjust(8) for i in range(5)]) + "   nb de plis ou XGBoost gagne")
for metrique in ["RMSE", "MAE", "MedAE", "NASA_moy", "Sur20_pct"]:
    differences = []
    for numero_pli in range(5):
        valeur_ridge = metriques_plis["Ridge"]["validation"][numero_pli][metrique]
        valeur_xgb = metriques_plis["XGBoost (46 features)"]["validation"][numero_pli][metrique]
        differences.append(valeur_ridge - valeur_xgb)
    nb_gagnes = int(np.sum(np.array(differences) > 0))
    texte = " ".join([f"{d:8.2f}" for d in differences])
    print(f"{metrique:10s} {texte}   {nb_gagnes}/5")

## 9. Chaque métrique : gardée ou écartée (protocole §9)

Chiffres = XGBoost, validation (moyenne ± écart-type entre plis) puis test officiel.

| Métrique | Verdict | Pourquoi (chiffres de ce notebook) |
|---|---|---|
| **MAE** | **Gardée** | Lisible directement : 11.49 ± 0.98 cycles en validation, 14.36 au test. Moins dominée par les grosses erreurs que le RMSE. |
| **MSE** | **Calculée, écartée du rapport** | C'est le RMSE au carré (286 ± 42 cycles²) : aucune information en plus, et une unité (cycles²) qu'on ne sait pas interpréter. Utile seulement comme fonction de perte. |
| **RMSE** | **Gardée** | 16.86 ± 1.25 en validation, 19.70 au test. Standard de la littérature C-MAPSS : indispensable pour se comparer. Plus sensible aux grosses erreurs que la MAE (16.86 contre 11.49). |
| **R²** | **Gardée** | 0.84 ± 0.02 (validation), 0.78 (test). Lecture immédiate par rapport aux baselines : 0.00 pour « Moyenne » par construction, négatif pour « Constante 125 » (−0.84) et « Aléatoire » (−1.11), c'est-à-dire pire que de prédire la moyenne. Réserve : le R² dépend de la variance du jeu évalué (cible plafonnée en validation, non plafonnée au test), donc validation et test ne se comparent pas directement. |
| **Erreur absolue médiane** | **Gardée** | 6.54 ± 0.72 contre une MAE de 11.49 : la moitié des lignes est prédite à moins de ~6.5 cycles, et quelques grosses erreurs tirent la moyenne vers le haut (au test : 11.08 contre 14.36). Écart que ni le RMSE ni la MAE ne montrent seuls. |
| **Erreur maximale** | **Gardée comme alerte, pas pour comparer** | 66.66 ± 3.68 en validation : un moteur peut être prédit à plus de 60 cycles de la vérité. Mais c'est un extrême, il dépend de la taille de l'échantillon (≈ 4 100 lignes par pli contre 100 moteurs au test : 59.08), donc très instable et inutilisable pour départager deux modèles. |
| **Biais moyen** | **Gardée, avec les tranches** | Global ≈ 0 (−0.07 ± 3.33 en validation, −0.90 au test) : XGBoost n'est pas biaisé en moyenne. Mais ce zéro masque des compensations : par tranche de RUL, +3.87 (≤ 30), **+10.30 (31-60)**, −3.19 (61-125) en validation. Le modèle surestime donc dans la zone où la dégradation commence, ce que le biais global ne dit pas. |
| **Score NASA, somme** | **Gardée pour le test seulement** | 852.9 au test (chiffre publié, comparable aux articles). En validation (29 729 ± 7 349) c'est une somme sur ≈ 4 100 lignes par pli : non comparable au test. |
| **Score NASA, moyenne par prédiction** | **Gardée** | Seule version comparable entre validation (7.24 ± 1.94) et test (8.53). Limite : Ridge fait 7.30 ± 1.69 en validation, indiscernable de XGBoost alors que le RMSE les sépare (19.56 contre 16.86). Le score NASA, exponentiel, est dominé par quelques grosses surestimations. |
| **Surestimations > 20 cycles** | **Gardée** | Lisible par un non-spécialiste : 11.6 % ± 2.9 des lignes en validation (≈ 475 par pli), 9 moteurs sur 100 au test (Ridge : 15.7 % et 20/100). Le seuil de 20 est arbitraire (pas dans le cahier des charges). |
| **MAPE** | **Écartée** | Valeur brute 6.3·10¹³ en validation : elle explose à cause des 100 lignes où la RUL vraie vaut 0 (le dernier cycle de chaque moteur ; diviser par 0). Même sans ces lignes, elle mesure surtout le niveau de la RUL : 0.82 pour RUL vraie de 1 à 10, 0.38 pour 11-30, 0.12 pour 61-125 : on « se trompe » en pourcentage surtout quand la RUL est petite, sans que l'erreur en cycles soit plus grande. Au test elle vaut 0.25 (aucune RUL vraie nulle), donc elle n'est même pas comparable à la validation. |

## 10. Ce que montrent les tableaux (faits, sans conclusion au-delà des chiffres)

- **Baselines.** Moyenne, médiane, constante 125 et aléatoire sont toutes à RMSE de validation ≥ 41.69 (R² ≤ 0.00). XGBoost (16.86) fait 2.5 fois mieux que « Moyenne » (le chiffre de la Phase 3 est reproduit). « Constante 125 » surestime de plus de 20 cycles sur 51 % des lignes (score NASA moyen 13 684 par prédiction).
- **Ridge.** Une régression linéaire sur les mêmes features atteint déjà R² = 0.78 ± 0.03 et un RMSE de 19.56 ± 1.21 en validation, 21.93 au test. XGBoost est meilleur de 2.7 cycles de RMSE en validation, et gagne sur les 5 plis en RMSE, MAE et médiane (section 8). Au score NASA moyen, la différence n'est pas établie (7.24 contre 7.30).
- **Où est l'erreur.** En validation, le RMSE de XGBoost est de 9.74 pour RUL ≤ 30, **22.04 pour 31-60** et 16.91 pour 61-125 : l'erreur n'est pas la plus forte tout près de la panne, mais dans la zone intermédiaire, où le modèle surestime en moyenne de 10.3 cycles. Au test, sur 25 moteurs de RUL ≤ 30, le RMSE est de 15.31, plus élevé qu'en validation (effectif faible).
- **Plafond.** Au test, pour 11 moteurs de RUL vraie > 125, XGBoost sous-estime de 28 cycles en moyenne : conséquence directe de l'entraînement sur une cible plafonnée à 125 (la figure de résidus le montre).
- **Écart train / validation.** RMSE 10.94 contre 16.86, R² 0.93 contre 0.84 : cohérent avec le diagnostic de la Phase 3 (traité plus à fond par une autre partie du protocole, §5).

## 11. Limites et points de méthode à connaître

- **Validation ≠ test.** Validation : toutes les lignes, cible plafonnée. Test : un point par moteur (dernier cycle), vérité non plafonnée, 100 points seulement. Les tranches du test ont des effectifs de 11 à 50 : leurs chiffres sont indicatifs.
- **Normalisation ajustée sur tout le train avant la validation croisée** (héritée de la Phase 3, protocole §1.3 demande de la refaire dans chaque pli). Pour les deux modèles de ce notebook, c'est sans effet : le min-max est une transformation linéaire croissante par colonne (les arbres de XGBoost coupent aux mêmes endroits), et le `StandardScaler` de Ridge, ajusté dans chaque pli, annule de toute façon cette échelle. Ce ne sera plus vrai pour un LSTM ou un CNN.
- **Pas de test de significativité** entre modèles (§8), ni d'optimisation des hyperparamètres, ni de courbes d'apprentissage, ni de métriques de classification : traités par d'autres parties du protocole.
- **Écart-type entre plis** : `np.std` avec ddof=0, comme en Phase 3 ; avec ddof=1 les valeurs seraient environ 12 % plus grandes.
- **XGBoost n'est pas strictement déterministe entre environnements** (voir `docs/03_modelisation.md`) : de très petits écarts de dernier chiffre sont possibles à la réexécution.
- **Baseline aléatoire** : un seul tirage par pli (graine 42 + numéro du pli) ; suffisant pour une borne basse, pas pour des statistiques fines.
- **MaxErr et Sur20_n sont affichés en moyenne de plis** (valeurs non entières possibles).